# Representation Analysis

Implements the full RepE methodology for multi-turn jailbreak conversations.

## Positions (Zhao et al.)
```
... [user content] <|eot_id|> <|start_header_id|> assistant <|end_header_id|> \n\n
         ↑               ↑
      t_inst        t_post_inst
```
- `h_inst` (t_inst): encodes *harmfulness* — last user token before EOT  
- `h_post` (t_post_inst): encodes *refusal* — the EOT token itself

## Behavioral categories
| Label | Condition |
|---|---|
| `accepted_harmful` | harmful + `attack_success=True` |
| `refused_harmful` | harmful + `attack_success=False` |
| `accepted_harmless` | benign + `attack_success=True` (verdict=jailbroken) |
| `refused_harmless` | benign + `attack_success=False` |

## Centroid options
| Option | Description |
|---|---|
| **A** | Final accepted turn only |
| **B** | Mean across all accepted turns (trajectory average) |
| **C** | Per-position: early (k=1), mid (k≈n/2), late (k=n) |

## Train / test split
- Train: `attempt ≤ 16` (reps 1–16)
- Test:  `attempt > 16` (reps 17–20)

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm

repo_root = Path("..").resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

REPR_ROOT = repo_root / "data" / "representations"
FRAMEWORKS = ["crescendo", "actorattack", "xteaming"]
SPLITS      = ["harmful", "benign"]
N_LAYERS    = 32
HIDDEN_DIM  = 4096

TRAIN_MAX_ATTEMPT = 16   # attempts 1–16 → train
# test: attempts 17–20

CATEGORY_COLORS = {
    "accepted_harmful":  "#d62728",
    "refused_harmful":   "#ff9896",
    "accepted_harmless": "#1f77b4",
    "refused_harmless":  "#aec7e8",
}

print(f"Repo root: {repo_root}")
print(f"REPR_ROOT: {REPR_ROOT}")

In [ ]:
# ── Load all trajectory data ──────────────────────────────────────────────────
# Each dataset: flat rows matching npy array positions.
# Keys in traj dict: (framework, split) → {"meta": DataFrame, "h_inst": (N, 32, 4096), "h_post": (N, 32, 4096)}

traj = {}

for fw in FRAMEWORKS:
    for split in SPLITS:
        folder = REPR_ROOT / "trajectories" / f"{fw}_{split}"
        meta   = pd.read_parquet(folder / "metadata.parquet")
        h_inst = np.load(folder / "h_inst.npy")       # float16, (N, 32, 4096)
        h_post = np.load(folder / "h_post_inst.npy")  # float16, (N, 32, 4096)
        assert len(meta) == len(h_inst) == len(h_post)
        traj[(fw, split)] = {"meta": meta, "h_inst": h_inst, "h_post": h_post}
        print(f"{fw:15s} {split:8s}  rows={len(meta):6d}  "
              f"convs={meta['conversation_id'].nunique():5d}")

# ── Single-turn baseline ──────────────────────────────────────────────────────
st = {}
for split in SPLITS:
    folder = REPR_ROOT / "single_turn" / split
    meta   = pd.read_parquet(folder / "metadata.parquet")
    h_inst = np.load(folder / "h_inst.npy")       # (100, 32, 4096)
    h_post = np.load(folder / "h_post_inst.npy")
    st[split] = {"meta": meta, "h_inst": h_inst, "h_post": h_post}
    print(f"single_turn   {split:8s}  rows={len(meta):6d}")

print("\nDone loading.")

In [ ]:
# ── Assign behavioral categories & train/test split ──────────────────────────
# Adds 'category' and 'split' columns to each metadata DataFrame (in-place).

def assign_category(row):
    """Map goal_type × attack_success → behavioral category."""
    if row["goal_type"] == "harmful":
        return "accepted_harmful" if row["attack_success"] else "refused_harmful"
    else:  # benign — attack_success is True iff verdict==jailbroken
        return "accepted_harmless" if row["attack_success"] else "refused_harmless"


for key, d in traj.items():
    m = d["meta"]
    m["category"] = m.apply(assign_category, axis=1)
    m["data_split"] = np.where(m["attempt"] <= TRAIN_MAX_ATTEMPT, "train", "test")

# ── Summary ───────────────────────────────────────────────────────────────────
print(f"{'Framework':<14} {'Split':<8} {'Train convs':>12} {'Test convs':>11}  Category breakdown")
print("-" * 80)
for fw in FRAMEWORKS:
    for split in SPLITS:
        m = traj[(fw, split)]["meta"]
        # Per-conversation stats (one row per conversation = last-turn row)
        convs = m.groupby("conversation_id").last().reset_index()
        train_n = (convs["data_split"] == "train").sum()
        test_n  = (convs["data_split"] == "test").sum()
        cats    = convs["category"].value_counts().to_dict()
        print(f"{fw:<14} {split:<8} {train_n:>12} {test_n:>11}  {cats}")

In [ ]:
# ── Representation extraction helpers (Options A / B / C) ────────────────────

def option_a(h_array, meta, data_split=None, category=None):
    """
    Option A: final accepted turn per conversation.
    Returns (h, conv_meta) where h is (N_convs, 32, 4096) float32
    and conv_meta is a DataFrame with one row per conversation.
    """
    m = meta.copy()
    if data_split is not None:
        m = m[m["data_split"] == data_split]
    if category is not None:
        if isinstance(category, str):
            category = [category]
        m = m[m["category"].isin(category)]
    # Last turn per conversation
    idx = m.groupby("conversation_id")["turn_k"].idxmax()
    rows = idx.values
    # rows are index labels — need positional indices
    pos  = meta.index.get_indexer(rows)
    h    = h_array[pos].astype(np.float32)    # (N, 32, 4096)
    conv_meta = meta.loc[rows].reset_index(drop=True)
    return h, conv_meta


def option_b(h_array, meta, data_split=None, category=None):
    """
    Option B: mean across all accepted turns per conversation.
    Returns (h, conv_meta) — h is (N_convs, 32, 4096) float32.
    conv_meta has one row per conversation (from last turn).
    """
    m = meta.copy()
    if data_split is not None:
        m = m[m["data_split"] == data_split]
    if category is not None:
        if isinstance(category, str):
            category = [category]
        m = m[m["category"].isin(category)]
    h_list, meta_list = [], []
    for cid, grp in m.groupby("conversation_id", sort=False):
        pos = meta.index.get_indexer(grp.index)
        h_list.append(h_array[pos].astype(np.float32).mean(axis=0))  # (32, 4096)
        meta_list.append(grp.iloc[-1])
    h = np.stack(h_list)                        # (N, 32, 4096)
    conv_meta = pd.DataFrame(meta_list).reset_index(drop=True)
    return h, conv_meta


def option_c(h_array, meta, data_split=None, category=None):
    """
    Option C: three per-position slices — early (k=1), mid (k≈n/2), late (k=n).
    Returns dict {"early": (h, meta), "mid": (h, meta), "late": (h, meta)}.
    Conversations with n_accepted_turns==1 appear in early and late but not mid.
    """
    m = meta.copy()
    if data_split is not None:
        m = m[m["data_split"] == data_split]
    if category is not None:
        if isinstance(category, str):
            category = [category]
        m = m[m["category"].isin(category)]

    result = {}
    for pos_name in ["early", "mid", "late"]:
        rows, meta_rows = [], []
        for cid, grp in m.groupby("conversation_id", sort=False):
            grp_s = grp.sort_values("turn_k")
            n = len(grp_s)
            if pos_name == "early":
                row = grp_s.iloc[0]
            elif pos_name == "mid":
                if n < 2:
                    continue  # skip 1-turn conversations
                row = grp_s.iloc[n // 2]
            else:  # late
                row = grp_s.iloc[-1]
            rows.append(meta.index.get_loc(row.name))
            meta_rows.append(row)
        h = h_array[rows].astype(np.float32)
        result[pos_name] = (h, pd.DataFrame(meta_rows).reset_index(drop=True))
    return result


print("Centroid helpers defined.")
# Quick sanity check on crescendo_harmful
key = ("crescendo", "harmful")
h_a, m_a = option_a(traj[key]["h_inst"], traj[key]["meta"])
h_b, m_b = option_b(traj[key]["h_inst"], traj[key]["meta"])
print(f"Option A crescendo_harmful: h={h_a.shape}, convs={len(m_a)}")
print(f"Option B crescendo_harmful: h={h_b.shape}, convs={len(m_b)}")

In [ ]:
# ── Topic-cancelled harmfulness direction v_harmful ───────────────────────────
#
# For each JBB pair k (pair_id = 0..99):
#   μ_h[k, L] = mean h_inst over ALL train harmful convs with pair_id=k  (Option A)
#   μ_b[k, L] = mean h_inst over ALL train benign  convs with pair_id=k  (Option A)
#   d[k, L]   = μ_h[k, L] − μ_b[k, L]
#
# v_harmful[L] = normalize( mean_k( d[k, L] ) )   shape (32, 4096)
#
# We compute this for each framework separately and then also pool across frameworks.

def compute_v_harmful(framework, option_fn=option_a):
    """Returns v_harmful: (32, 4096) unit-normalised float32."""
    h_harm, m_harm = option_fn(
        traj[(framework, "harmful")]["h_inst"],
        traj[(framework, "harmful")]["meta"],
        data_split="train",
    )
    h_beni, m_beni = option_fn(
        traj[(framework, "benign")]["h_inst"],
        traj[(framework, "benign")]["meta"],
        data_split="train",
    )

    # Per-pair mean — (100, 32, 4096)
    pairs = sorted(m_harm["pair_id"].unique())
    d_list = []
    for k in pairs:
        idx_h = m_harm.index[m_harm["pair_id"] == k].tolist()
        idx_b = m_beni.index[m_beni["pair_id"] == k].tolist()
        if not idx_h or not idx_b:
            continue
        mu_h = h_harm[idx_h].mean(axis=0)   # (32, 4096)
        mu_b = h_beni[idx_b].mean(axis=0)   # (32, 4096)
        d_list.append(mu_h - mu_b)

    d_mean = np.stack(d_list).mean(axis=0)  # (32, 4096)

    # Normalise per layer
    norms = np.linalg.norm(d_mean, axis=-1, keepdims=True)  # (32, 1)
    v = d_mean / (norms + 1e-12)
    return v   # (32, 4096)


v_harmful = {}
for fw in FRAMEWORKS:
    v_harmful[fw] = compute_v_harmful(fw)
    print(f"v_harmful[{fw}]: shape={v_harmful[fw].shape}  "
          f"norm_layer16={np.linalg.norm(v_harmful[fw][16]):.4f}")

# Pooled direction: average across frameworks then re-normalise
d_pooled = np.stack([v_harmful[fw] for fw in FRAMEWORKS]).mean(axis=0)
v_harmful["pooled"] = d_pooled / (np.linalg.norm(d_pooled, axis=-1, keepdims=True) + 1e-12)
print(f"v_harmful[pooled]: shape={v_harmful['pooled'].shape}")

In [ ]:
# ── Refusal direction v_refusal ───────────────────────────────────────────────
#
# Using h_post_inst (t_post_inst = EOT token), from harmful train conversations.
# v_refusal[L] = normalize( mean(h_post of refused_harmful) − mean(h_post of accepted_harmful) )
#
# Computed per framework (Option A — final turn).

def compute_v_refusal(framework, option_fn=option_a):
    """Returns v_refusal: (32, 4096) unit-normalised float32."""
    h_ref, m_ref = option_fn(
        traj[(framework, "harmful")]["h_post"],
        traj[(framework, "harmful")]["meta"],
        data_split="train",
        category="refused_harmful",
    )
    h_acc, m_acc = option_fn(
        traj[(framework, "harmful")]["h_post"],
        traj[(framework, "harmful")]["meta"],
        data_split="train",
        category="accepted_harmful",
    )

    mu_ref = h_ref.mean(axis=0)   # (32, 4096)
    mu_acc = h_acc.mean(axis=0)   # (32, 4096)
    diff   = mu_ref - mu_acc
    norms  = np.linalg.norm(diff, axis=-1, keepdims=True)
    v = diff / (norms + 1e-12)
    return v  # (32, 4096)


v_refusal = {}
for fw in FRAMEWORKS:
    v_refusal[fw] = compute_v_refusal(fw)
    print(f"v_refusal[{fw}]: shape={v_refusal[fw].shape}")

d_ref_pooled = np.stack([v_refusal[fw] for fw in FRAMEWORKS]).mean(axis=0)
v_refusal["pooled"] = d_ref_pooled / (np.linalg.norm(d_ref_pooled, axis=-1, keepdims=True) + 1e-12)
print(f"v_refusal[pooled]: shape={v_refusal['pooled'].shape}")

In [ ]:
# ── Single-turn baseline directions ──────────────────────────────────────────
#
# No attack context. Replicates Zhao et al.'s Latent Guard baseline.
#
# v_harmful_st[L] = normalize( mean_k( h_inst[harmful, k] − h_inst[benign, k] ) )
# v_refusal_st[L] = not meaningful for single-turn (no refusal label),
#                   so we skip it here.

h_st_h = st["harmful"]["h_inst"].astype(np.float32)  # (100, 32, 4096)
h_st_b = st["benign"]["h_inst"].astype(np.float32)   # (100, 32, 4096)
m_st_h = st["harmful"]["meta"]
m_st_b = st["benign"]["meta"]

# Align by pair_id
h_st_h_sorted = h_st_h[m_st_h.sort_values("pair_id").index]  # (100, 32, 4096)
h_st_b_sorted = h_st_b[m_st_b.sort_values("pair_id").index]

d_st = h_st_h_sorted - h_st_b_sorted                 # (100, 32, 4096)
d_st_mean = d_st.mean(axis=0)                          # (32, 4096)
norms_st = np.linalg.norm(d_st_mean, axis=-1, keepdims=True)
v_harmful_st = d_st_mean / (norms_st + 1e-12)          # (32, 4096)

print(f"v_harmful_st: {v_harmful_st.shape}")

In [ ]:
# ── Direction cosine similarities across layers ────────────────────────────────
#
# 1. v_harmful per framework vs v_harmful_st (single-turn) — per layer
# 2. v_harmful vs v_refusal — per layer, per framework
# 3. Cross-framework: cos(v_harmful[fw_i], v_harmful[fw_j]) per layer

def cosine_per_layer(va, vb):
    """va, vb: (32, 4096) normalised. Returns (32,) cosine similarities."""
    return (va * vb).sum(axis=-1)  # already unit vectors


layers = np.arange(1, N_LAYERS + 1)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# ── Panel 1: multi-turn v_harmful vs single-turn v_harmful_st ─────────────────
ax = axes[0]
for fw in FRAMEWORKS:
    cos = cosine_per_layer(v_harmful[fw], v_harmful_st)
    ax.plot(layers, cos, label=fw)
ax.axhline(0, color="gray", lw=0.5)
ax.set_xlabel("Layer")
ax.set_ylabel("Cosine similarity")
ax.set_title("v_harmful (multi-turn) vs v_harmful_st (baseline)")
ax.legend()
ax.set_ylim(-1.05, 1.05)

# ── Panel 2: v_harmful vs v_refusal (same framework) ─────────────────────────
ax = axes[1]
for fw in FRAMEWORKS:
    cos = cosine_per_layer(v_harmful[fw], v_refusal[fw])
    ax.plot(layers, cos, label=fw)
ax.axhline(0, color="gray", lw=0.5)
ax.set_xlabel("Layer")
ax.set_title("v_harmful vs v_refusal (same framework)")
ax.legend()
ax.set_ylim(-1.05, 1.05)

# ── Panel 3: cross-framework cosine for v_harmful ────────────────────────────
ax = axes[2]
pairs_fw = [("crescendo", "actorattack"), ("crescendo", "xteaming"), ("actorattack", "xteaming")]
for fa, fb in pairs_fw:
    cos = cosine_per_layer(v_harmful[fa], v_harmful[fb])
    ax.plot(layers, cos, label=f"{fa} ↔ {fb}")
ax.axhline(0, color="gray", lw=0.5)
ax.set_xlabel("Layer")
ax.set_title("v_harmful: cross-framework cosine")
ax.legend(fontsize=8)
ax.set_ylim(-1.05, 1.05)

plt.tight_layout()
plt.savefig("../figures/04_direction_cosines.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved figures/04_direction_cosines.png")

In [ ]:
# ── Category projections on test set ─────────────────────────────────────────
#
# Project h_inst (Option A, test set) onto v_harmful and h_post onto v_refusal.
# Show violin plots per category for a chosen layer.
# Also compute AUROC (accepted_harmful vs others) per layer.

FOCAL_LAYER = 16   # 0-indexed → layer 17 in 1-indexed display

def project(h, direction, layer):
    """h: (N, 32, 4096), direction: (32, 4096). Returns (N,) projections at `layer`."""
    return (h[:, layer, :].astype(np.float32) @ direction[layer].astype(np.float32))


# ── Per-framework projection plots at FOCAL_LAYER ─────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for col, fw in enumerate(FRAMEWORKS):
    # h_inst projections
    h_test, m_test = option_a(
        traj[(fw, "harmful")]["h_inst"], traj[(fw, "harmful")]["meta"],
        data_split="test",
    )
    h_test_b, m_test_b = option_a(
        traj[(fw, "benign")]["h_inst"], traj[(fw, "benign")]["meta"],
        data_split="test",
    )
    # Combine
    h_all   = np.concatenate([h_test,  h_test_b], axis=0)
    m_all   = pd.concat([m_test, m_test_b], ignore_index=True)
    proj_h  = project(h_all, v_harmful[fw], FOCAL_LAYER)

    ax = axes[0, col]
    cats = ["accepted_harmful", "refused_harmful", "accepted_harmless", "refused_harmless"]
    data_v = [proj_h[m_all["category"] == c] for c in cats]
    parts  = ax.violinplot(data_v, positions=range(4), showmedians=True)
    for i, (pc, c) in enumerate(zip(parts["bodies"], cats)):
        pc.set_facecolor(CATEGORY_COLORS[c])
        pc.set_alpha(0.75)
    ax.set_xticks(range(4))
    ax.set_xticklabels([c.replace("_", "\n") for c in cats], fontsize=8)
    ax.set_title(f"{fw} — v_harmful proj (layer {FOCAL_LAYER+1})")
    if col == 0:
        ax.set_ylabel("Projection")

    # h_post projections
    h_post_test, _ = option_a(
        traj[(fw, "harmful")]["h_post"], traj[(fw, "harmful")]["meta"],
        data_split="test",
    )
    h_post_test_b, _ = option_a(
        traj[(fw, "benign")]["h_post"], traj[(fw, "benign")]["meta"],
        data_split="test",
    )
    hp_all  = np.concatenate([h_post_test, h_post_test_b], axis=0)
    proj_r  = project(hp_all, v_refusal[fw], FOCAL_LAYER)

    ax = axes[1, col]
    data_v = [proj_r[m_all["category"] == c] for c in cats]
    parts  = ax.violinplot(data_v, positions=range(4), showmedians=True)
    for i, (pc, c) in enumerate(zip(parts["bodies"], cats)):
        pc.set_facecolor(CATEGORY_COLORS[c])
        pc.set_alpha(0.75)
    ax.set_xticks(range(4))
    ax.set_xticklabels([c.replace("_", "\n") for c in cats], fontsize=8)
    ax.set_title(f"{fw} — v_refusal proj (layer {FOCAL_LAYER+1})")
    if col == 0:
        ax.set_ylabel("Projection")

plt.suptitle(f"Category projections — test set (Option A, layer {FOCAL_LAYER+1})", y=1.01)
plt.tight_layout()
plt.savefig("../figures/04_category_projections.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved figures/04_category_projections.png")

In [ ]:
# ── AUROC per layer ───────────────────────────────────────────────────────────
#
# Two tasks evaluated at each layer:
#   (1) v_harmful — h_inst, harmful vs benign (test set)          [harmfulness signal]
#   (2) v_refusal — h_post, refused_harmful vs accepted_harmful   [refusal signal]
#
# For (1): also compare crescendo direction vs same-framework vs single-turn baseline.

def auroc_per_layer(h_pos, h_neg, direction):
    """
    Compute AUROC at each layer.
    h_pos / h_neg: (N, 32, 4096). direction: (32, 4096).
    Returns (32,) float array.
    """
    aucs = []
    for l in range(N_LAYERS):
        p_proj = h_pos[:, l, :].astype(np.float32) @ direction[l].astype(np.float32)
        n_proj = h_neg[:, l, :].astype(np.float32) @ direction[l].astype(np.float32)
        scores = np.concatenate([p_proj, n_proj])
        labels = np.concatenate([np.ones(len(p_proj)), np.zeros(len(n_proj))])
        try:
            aucs.append(roc_auc_score(labels, scores))
        except ValueError:
            aucs.append(float("nan"))
    return np.array(aucs)


fig, axes = plt.subplots(2, 3, figsize=(16, 8))

for col, fw in enumerate(FRAMEWORKS):
    # ── Row 0: v_harmful — harmful vs benign ─────────────────────────────────
    ax = axes[0, col]
    h_harm_te, _ = option_a(
        traj[(fw, "harmful")]["h_inst"], traj[(fw, "harmful")]["meta"],
        data_split="test",
    )
    h_beni_te, _ = option_a(
        traj[(fw, "benign")]["h_inst"], traj[(fw, "benign")]["meta"],
        data_split="test",
    )

    auc_vh_own = auroc_per_layer(h_harm_te, h_beni_te, v_harmful[fw])
    auc_vh_cre = auroc_per_layer(h_harm_te, h_beni_te, v_harmful["crescendo"])
    auc_st     = auroc_per_layer(h_harm_te, h_beni_te, v_harmful_st)

    ax.plot(layers, auc_vh_own, label=f"v_harmful ({fw})",           color="#d62728")
    ax.plot(layers, auc_vh_cre, label="v_harmful (crescendo, xfer)", color="#ff7f0e", ls="--")
    ax.plot(layers, auc_st,     label="v_harmful_st (baseline)",     color="#2ca02c", ls=":")
    ax.axhline(0.5, color="gray", lw=0.5, ls="--")
    ax.set_xlabel("Layer")
    ax.set_ylim(0.4, 1.0)
    ax.set_title(f"{fw} — v_harmful AUROC (harmful vs benign)")
    ax.legend(fontsize=7)
    if col == 0:
        ax.set_ylabel("AUROC")

    # ── Row 1: v_refusal — refused_harmful vs accepted_harmful ────────────────
    ax = axes[1, col]
    h_ref_te, _ = option_a(
        traj[(fw, "harmful")]["h_post"], traj[(fw, "harmful")]["meta"],
        data_split="test", category="refused_harmful",
    )
    h_acc_te, _ = option_a(
        traj[(fw, "harmful")]["h_post"], traj[(fw, "harmful")]["meta"],
        data_split="test", category="accepted_harmful",
    )

    auc_vr_own = auroc_per_layer(h_ref_te, h_acc_te, v_refusal[fw])
    auc_vr_cre = auroc_per_layer(h_ref_te, h_acc_te, v_refusal["crescendo"])

    ax.plot(layers, auc_vr_own, label=f"v_refusal ({fw})",           color="#1f77b4")
    ax.plot(layers, auc_vr_cre, label="v_refusal (crescendo, xfer)", color="#aec7e8", ls="--")
    ax.axhline(0.5, color="gray", lw=0.5, ls="--")
    ax.set_xlabel("Layer")
    ax.set_ylim(0.4, 1.0)
    ax.set_title(f"{fw} — v_refusal AUROC (refused vs accepted harmful)")
    ax.legend(fontsize=7)
    if col == 0:
        ax.set_ylabel("AUROC")

plt.tight_layout()
plt.savefig("../figures/04_auroc_per_layer.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved figures/04_auroc_per_layer.png")

In [ ]:
# ── Trajectory projections — how does the signal evolve across turns? (RQ2) ──
#
# For each conversation in the test set, compute projection at each turn k.
# Show mean ± SEM for accepted_harmful vs refused_harmful.
# One plot per framework, using crescendo's v_harmful direction (cross-framework test).

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for col, fw in enumerate(FRAMEWORKS):
    ax = axes[col]
    meta_h  = traj[(fw, "harmful")]["meta"]
    h_inst  = traj[(fw, "harmful")]["h_inst"]
    v_dir   = v_harmful["crescendo"][FOCAL_LAYER]  # (4096,)

    # Test set only
    test_meta = meta_h[meta_h["data_split"] == "test"].copy()

    for cat, color in [("accepted_harmful", "#d62728"), ("refused_harmful", "#ff9896")]:
        sub = test_meta[test_meta["category"] == cat]
        max_turn = int(sub["turn_k"].max())
        means, sems = [], []
        turn_ns = []
        for k in range(1, max_turn + 1):
            rows = sub[sub["turn_k"] == k]
            if len(rows) < 3:
                break
            pos  = meta_h.index.get_indexer(rows.index)
            proj = h_inst[pos][:, FOCAL_LAYER, :].astype(np.float32) @ v_dir
            means.append(proj.mean())
            sems.append(proj.std() / np.sqrt(len(proj)))
            turn_ns.append(k)
        means = np.array(means)
        sems  = np.array(sems)
        ax.plot(turn_ns, means, color=color, label=cat.replace("_", " "), marker="o", ms=4)
        ax.fill_between(turn_ns, means - sems, means + sems, color=color, alpha=0.2)

    ax.set_xlabel("Turn k")
    ax.set_title(f"{fw} — v_harmful projection vs turn (layer {FOCAL_LAYER+1})")
    ax.legend(fontsize=8)
    if col == 0:
        ax.set_ylabel("Projection")

plt.tight_layout()
plt.savefig("../figures/04_trajectory_projections.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved figures/04_trajectory_projections.png")

In [ ]:
# ── Option A vs B vs C comparison ────────────────────────────────────────────
#
# Compare AUROC (accepted_harmful vs refused_harmful) at each layer
# for three centroid aggregation methods.

COMP_FW = "crescendo"  # framework to illustrate

# Option A — final turn direction
v_harm_a = compute_v_harmful(COMP_FW, option_fn=option_a)
# Option B — trajectory-mean direction
v_harm_b = compute_v_harmful(COMP_FW, option_fn=option_b)

# Option C — direction from final-turn h (= option_a), but compute separately
# early/mid/late directions and compare
def option_a_pos(h_array, meta, data_split=None, position="early"):
    """Option A variant restricted to a specific trajectory position."""
    c_result = option_c(h_array, meta, data_split=data_split)
    return c_result[position]


def compute_v_harmful_pos(framework, position):
    """Like compute_v_harmful but using only a specific turn position."""
    def _opt_fn(h_array, meta, data_split=None, category=None):
        h, m = option_c(h_array, meta, data_split=data_split)[position]
        if category is not None:
            if isinstance(category, str):
                category_list = [category]
            else:
                category_list = category
            mask = m["category"].isin(category_list)
            h = h[mask]
            m = m[mask].reset_index(drop=True)
        return h, m
    return compute_v_harmful(framework, option_fn=_opt_fn)


v_harm_c_early = compute_v_harmful_pos(COMP_FW, "early")
v_harm_c_late  = compute_v_harmful_pos(COMP_FW, "late")

# Evaluate AUROC on test set
h_acc, _ = option_a(
    traj[(COMP_FW, "harmful")]["h_inst"], traj[(COMP_FW, "harmful")]["meta"],
    data_split="test", category="accepted_harmful",
)
h_ref, _ = option_a(
    traj[(COMP_FW, "harmful")]["h_inst"], traj[(COMP_FW, "harmful")]["meta"],
    data_split="test", category="refused_harmful",
)

auc_a      = auroc_per_layer(h_acc, h_ref, v_harm_a)
auc_b      = auroc_per_layer(h_acc, h_ref, v_harm_b)
auc_c_e    = auroc_per_layer(h_acc, h_ref, v_harm_c_early)
auc_c_l    = auroc_per_layer(h_acc, h_ref, v_harm_c_late)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(layers, auc_a,   label="Option A (final turn)",    color="#1f77b4")
ax.plot(layers, auc_b,   label="Option B (traj. mean)",    color="#ff7f0e", ls="--")
ax.plot(layers, auc_c_e, label="Option C-early (turn 1)",  color="#2ca02c", ls=":")
ax.plot(layers, auc_c_l, label="Option C-late (turn n)",   color="#d62728", ls="-.")
ax.axhline(0.5, color="gray", lw=0.5, ls="--")
ax.set_xlabel("Layer")
ax.set_ylabel("AUROC")
ax.set_title(f"{COMP_FW} — centroid option comparison (acc. harmful vs ref. harmful)")
ax.legend()
ax.set_ylim(0.4, 1.0)
plt.tight_layout()
plt.savefig("../figures/04_option_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved figures/04_option_comparison.png")

In [ ]:
# ── MLP probe — goal-stratified CV ───────────────────────────────────────────
#
# Task: binary classification — harmful (1) vs benign (0), final turn (Option A)
# Stratification: pair_id as group → GroupKFold(n_splits=5)
#   prevents probe from memorising topic; tests goal-level generalisation.
#
# Train on train set. Reports mean AUROC ± std across folds per layer.
# Then evaluates on the held-out test set using a full-train-set-fitted probe.

def mlp_probe_auroc(framework, n_splits=5):
    """Returns (mean_auc_cv, std_auc_cv, test_auc) arrays of length N_LAYERS."""
    h_harm_tr, m_harm_tr = option_a(
        traj[(framework, "harmful")]["h_inst"],
        traj[(framework, "harmful")]["meta"],
        data_split="train",
    )
    h_beni_tr, m_beni_tr = option_a(
        traj[(framework, "benign")]["h_inst"],
        traj[(framework, "benign")]["meta"],
        data_split="train",
    )
    h_harm_te, _ = option_a(
        traj[(framework, "harmful")]["h_inst"],
        traj[(framework, "harmful")]["meta"],
        data_split="test",
    )
    h_beni_te, _ = option_a(
        traj[(framework, "benign")]["h_inst"],
        traj[(framework, "benign")]["meta"],
        data_split="test",
    )

    X_tr  = np.concatenate([h_harm_tr, h_beni_tr], axis=0)
    y_tr  = np.concatenate([np.ones(len(h_harm_tr)), np.zeros(len(h_beni_tr))]).astype(int)
    grp   = np.concatenate([m_harm_tr["pair_id"].values, m_beni_tr["pair_id"].values])

    X_te  = np.concatenate([h_harm_te, h_beni_te], axis=0)
    y_te  = np.concatenate([np.ones(len(h_harm_te)), np.zeros(len(h_beni_te))]).astype(int)

    gkf = GroupKFold(n_splits=n_splits)
    mean_cv, std_cv, test_auc = [], [], []

    for layer in tqdm(range(N_LAYERS), desc=f"MLP probe {framework}", leave=False):
        X_l_tr = X_tr[:, layer, :].astype(np.float32)
        X_l_te = X_te[:, layer, :].astype(np.float32)
        fold_aucs = []

        for tr_idx, val_idx in gkf.split(X_l_tr, y_tr, groups=grp):
            scaler = StandardScaler()
            X_f_tr = scaler.fit_transform(X_l_tr[tr_idx])
            X_f_val = scaler.transform(X_l_tr[val_idx])
            if len(np.unique(y_tr[val_idx])) < 2:
                continue
            clf = MLPClassifier(hidden_layer_sizes=(256,), max_iter=200,
                                random_state=42, early_stopping=True)
            clf.fit(X_f_tr, y_tr[tr_idx])
            fold_aucs.append(roc_auc_score(y_tr[val_idx], clf.predict_proba(X_f_val)[:, 1]))

        # Test AUC: fit on full train, eval on held-out test
        scaler_full = StandardScaler()
        clf_full = MLPClassifier(hidden_layer_sizes=(256,), max_iter=200,
                                 random_state=42, early_stopping=True)
        clf_full.fit(scaler_full.fit_transform(X_l_tr), y_tr)
        t_auc = roc_auc_score(y_te, clf_full.predict_proba(scaler_full.transform(X_l_te))[:, 1])

        mean_cv.append(np.mean(fold_aucs) if fold_aucs else float("nan"))
        std_cv.append(np.std(fold_aucs) if fold_aucs else float("nan"))
        test_auc.append(t_auc)

    return np.array(mean_cv), np.array(std_cv), np.array(test_auc)


probe_results = {}
for fw in FRAMEWORKS:
    probe_results[fw] = mlp_probe_auroc(fw)

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
colors = ["#1f77b4", "#ff7f0e", "#2ca02c"]

for ax, which in zip(axes, ["cv", "test"]):
    for fw, color in zip(FRAMEWORKS, colors):
        mean_cv, std_cv, test_auc = probe_results[fw]
        if which == "cv":
            ax.plot(layers, mean_cv, label=fw, color=color)
            ax.fill_between(layers, mean_cv - std_cv, mean_cv + std_cv, color=color, alpha=0.15)
        else:
            ax.plot(layers, test_auc, label=fw, color=color)
    ax.axhline(0.5, color="gray", lw=0.5, ls="--")
    ax.set_xlabel("Layer")
    ax.set_ylim(0.4, 1.05)
    title = "MLP probe: CV AUROC (goal-stratified, harmful vs benign)" if which == "cv" \
            else "MLP probe: test set AUROC (harmful vs benign)"
    ax.set_title(title)
    ax.set_ylabel("AUROC")
    ax.legend()

plt.tight_layout()
plt.savefig("../figures/04_mlp_probe.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved figures/04_mlp_probe.png")

In [ ]:
# ── Cross-framework transfer matrix ─────────────────────────────────────────
#
# For each pair (train_fw, test_fw):
#   Train v_harmful direction on train_fw train set.
#   Evaluate AUROC (harmful vs benign) on test_fw test set.
#   Report at the best layer and at FOCAL_LAYER.
#
# A strong off-diagonal means the direction is content-agnostic and framework-agnostic.

auc_matrix_focal  = np.zeros((3, 3))
auc_matrix_best   = np.zeros((3, 3))
best_layer_matrix = np.zeros((3, 3), dtype=int)

for i, train_fw in enumerate(FRAMEWORKS):
    for j, test_fw in enumerate(FRAMEWORKS):
        h_harm_te, _ = option_a(
            traj[(test_fw, "harmful")]["h_inst"],
            traj[(test_fw, "harmful")]["meta"],
            data_split="test",
        )
        h_beni_te, _ = option_a(
            traj[(test_fw, "benign")]["h_inst"],
            traj[(test_fw, "benign")]["meta"],
            data_split="test",
        )

        auc_l = auroc_per_layer(h_harm_te, h_beni_te, v_harmful[train_fw])
        auc_matrix_focal[i, j]  = auc_l[FOCAL_LAYER]
        best_l = int(np.nanargmax(auc_l))
        auc_matrix_best[i, j]   = auc_l[best_l]
        best_layer_matrix[i, j] = best_l + 1  # 1-indexed


fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, mat, title in zip(
    axes,
    [auc_matrix_focal, auc_matrix_best],
    [f"AUROC at layer {FOCAL_LAYER+1}", "AUROC at best layer"],
):
    im = ax.imshow(mat, vmin=0.5, vmax=1.0, cmap="RdYlGn")
    ax.set_xticks(range(3))
    ax.set_yticks(range(3))
    ax.set_xticklabels(FRAMEWORKS, rotation=30, ha="right")
    ax.set_yticklabels(FRAMEWORKS)
    ax.set_xlabel("Test framework")
    ax.set_ylabel("Train framework (v_harmful)")
    ax.set_title(title)
    for ii in range(3):
        for jj in range(3):
            val = mat[ii, jj]
            lbl = f"{val:.3f}"
            if mat is auc_matrix_best:
                lbl += f"\n(L{best_layer_matrix[ii,jj]})"
            ax.text(jj, ii, lbl, ha="center", va="center", fontsize=8,
                    color="black" if val < 0.85 else "white")
    plt.colorbar(im, ax=ax)

plt.suptitle("Cross-framework transfer: v_harmful AUROC (harmful vs benign)")
plt.tight_layout()
plt.savefig("../figures/04_cross_framework_transfer.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved figures/04_cross_framework_transfer.png")
print("\nBest layer per (train, test) pair:")
print(pd.DataFrame(best_layer_matrix, index=FRAMEWORKS, columns=FRAMEWORKS).to_string())

In [ ]:
# ── Single-turn baseline comparison ─────────────────────────────────────────
#
# Compare v_harmful projection distributions between:
#   - Single-turn harmful (no attack context) — baseline
#   - Multi-turn accepted_harmful (at final turn)
#   - Multi-turn refused_harmful  (at final turn)
#
# Pooled across all 3 frameworks; uses crescendo v_harmful direction.

v_dir_focal = v_harmful["crescendo"][FOCAL_LAYER]

# Single-turn projections
proj_st_harm = h_st_h[:, FOCAL_LAYER, :].astype(np.float32) @ v_dir_focal
proj_st_beni = h_st_b[:, FOCAL_LAYER, :].astype(np.float32) @ v_dir_focal

# Multi-turn (test set, all frameworks pooled)
proj_mt_acc, proj_mt_ref = [], []
for fw in FRAMEWORKS:
    h_a, _ = option_a(
        traj[(fw, "harmful")]["h_inst"], traj[(fw, "harmful")]["meta"],
        data_split="test", category="accepted_harmful",
    )
    h_r, _ = option_a(
        traj[(fw, "harmful")]["h_inst"], traj[(fw, "harmful")]["meta"],
        data_split="test", category="refused_harmful",
    )
    proj_mt_acc.append(h_a[:, FOCAL_LAYER, :].astype(np.float32) @ v_dir_focal)
    proj_mt_ref.append(h_r[:, FOCAL_LAYER, :].astype(np.float32) @ v_dir_focal)

proj_mt_acc = np.concatenate(proj_mt_acc)
proj_mt_ref = np.concatenate(proj_mt_ref)

fig, ax = plt.subplots(figsize=(8, 4))
bins = np.linspace(
    min(proj_st_harm.min(), proj_mt_acc.min(), proj_mt_ref.min(), proj_st_beni.min()),
    max(proj_st_harm.max(), proj_mt_acc.max(), proj_mt_ref.max(), proj_st_beni.max()),
    50,
)
for data, label, color, ls in [
    (proj_st_harm, "single-turn harmful",     "#d62728", "-"),
    (proj_mt_acc,  "multi-turn acc. harmful", "#ff7f0e", "--"),
    (proj_mt_ref,  "multi-turn ref. harmful", "#ff9896", ":"),
    (proj_st_beni, "single-turn benign",       "#1f77b4", "-."),
]:
    ax.hist(data, bins=bins, alpha=0.5, label=label, color=color,
            density=True, histtype="stepfilled", ls=ls)

ax.set_xlabel(f"Projection onto v_harmful (layer {FOCAL_LAYER+1})")
ax.set_ylabel("Density")
ax.set_title("Single-turn baseline vs multi-turn: v_harmful projections")
ax.legend()
plt.tight_layout()
plt.savefig("../figures/04_single_vs_multi_turn.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved figures/04_single_vs_multi_turn.png")

In [ ]:
# ── Summary table ─────────────────────────────────────────────────────────────
# Per framework:
#   - v_harmful  peak AUROC: harmful vs benign (own direction, layer sweep)
#   - v_refusal  peak AUROC: refused vs accepted harmful
#   - MLP probe  peak AUROC: harmful vs benign (goal-stratified CV)

rows = []
for fw in FRAMEWORKS:
    # v_harmful: harmful vs benign
    h_harm_te, _ = option_a(
        traj[(fw, "harmful")]["h_inst"], traj[(fw, "harmful")]["meta"],
        data_split="test",
    )
    h_beni_te, _ = option_a(
        traj[(fw, "benign")]["h_inst"], traj[(fw, "benign")]["meta"],
        data_split="test",
    )
    auc_vh  = auroc_per_layer(h_harm_te, h_beni_te, v_harmful[fw])
    auc_xfr = auroc_per_layer(h_harm_te, h_beni_te, v_harmful["crescendo"])

    # v_refusal: refused vs accepted harmful
    h_ref_te, _ = option_a(
        traj[(fw, "harmful")]["h_post"], traj[(fw, "harmful")]["meta"],
        data_split="test", category="refused_harmful",
    )
    h_acc_te, _ = option_a(
        traj[(fw, "harmful")]["h_post"], traj[(fw, "harmful")]["meta"],
        data_split="test", category="accepted_harmful",
    )
    auc_vr = auroc_per_layer(h_ref_te, h_acc_te, v_refusal[fw])

    # MLP
    mean_auc_mlp, _ = probe_results[fw]

    rows.append({
        "framework":             fw,
        "n_test_harmful":        len(h_harm_te),
        "n_test_benign":         len(h_beni_te),
        "n_test_refused":        len(h_ref_te),
        "n_test_accepted":       len(h_acc_te),
        "v_harmful_peak_auc":    float(np.nanmax(auc_vh)),
        "v_harmful_peak_layer":  int(np.nanargmax(auc_vh)) + 1,
        "v_harmful_xfer_auc":    float(np.nanmax(auc_xfr)),
        "v_refusal_peak_auc":    float(np.nanmax(auc_vr)),
        "v_refusal_peak_layer":  int(np.nanargmax(auc_vr)) + 1,
        "mlp_peak_auc":          float(np.nanmax(mean_auc_mlp)),
        "mlp_peak_layer":        int(np.nanargmax(mean_auc_mlp)) + 1,
    })

df_summary = pd.DataFrame(rows)
display(df_summary.set_index("framework").round(3))